# ANU VSM — Screening and DCD export

Interactive notebook for preliminary processing of ANU VSM hysteresis and backfield measurements.

## Input and output

- **Input:** hysteresis and backfield measurement files (including extensionless ANU files). Magnetic field is expected in T and moment in A m²; specimen masses can be loaded from an Excel file.
- **Screening:** plots of raw and corrected hysteresis, backfield and derivative curves, Day plots, and preliminary coercivity parameters. Exports PDF/PNG figures and an Excel summary of Bcr. It does not export converted backfield files.
- **Backfield conversion:** two-column files for DCD unmixing, with field converted to mT and original signed moment retained unless mass normalization is selected. Converted files and tables are saved in separate folders under the selected output directory.

## Usage

1. Run the notebook cells and select **Screening** or **Backfield conversion**.
2. Select the input folder, enter a specimen-ID example and specify measurement filename suffixes.
3. Scan the folder and review or correct the specimen/file mapping.
4. Optionally load masses from Excel; select the output folder and process the measurements.

Example raw measurements: `examples/vsm/TA04-05-500ms_500mT_hys` and `examples/vsm/TA04-05-5s_500mT_back` (paths relative to the repository root).

## Methods and limitations

Screening applies a simple high-field paramagnetic correction and is intended for a first assessment, not detailed hysteresis correction. Bcr estimates should be checked for noisy or multiple zero crossings. The Day plot includes reference curves digitized from Dunlop (2002). Existing output files with the same names may be overwritten.

## Requirements

`numpy`, `pandas`, `scipy`, `matplotlib`, `ipywidgets`, `openpyxl`. Optional: `ipyfilechooser` (folder browsing) and `xlrd` (older Excel files).

## References

Dunlop, D. J. (2002). Theory and application of the Day plot (Mrs/Ms versus Hcr/Hc) 1. Theoretical curves and tests using titanomagnetite data. *Journal of Geophysical Research: Solid Earth*, **107**, 2056. https://doi.org/10.1029/2001JB000486


## Numerical tools
Inherited scientific routines; edit interface settings rather than these functions for routine use.

In [ ]:
import glob
import os
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

# ==========================================
# CONFIGURATION & PATHS
# ==========================================
RAW_DATA_DIR = "Raw-data"
MASS_FILE = "mass.xls"
PARAMAG_CUTOFF_RATIO = 0.70  # Threshold for saturation fit (top 30% field)
OUTPUT_FILENAME = "VSM-summary.pdf"

# DCD export export
DCD_EXPORT_RAW_DIR = "DCD_Export_Raw"
DCD_EXPORT_SMOOTH_DIR = "DCD_Export_Smoothed"
DCD_EXPORT_START_FIELD_MT = 0.5
DCD_EXPORT_EXPORT_SUMMARY = "DCD_export_summary.csv"
BCR_OUTPUT_FILENAME = "Bcr_from_backfield.xlsx"

# ==========================================
# DUNLOP DAY-PLOT MIXING CURVES
# Embedded directly in the notebook, so no external CSV is required.
# Digitized from a figure in Dunlop (2002), doi:10.1029/2001JB000486.
# Each tuple contains (Bcr/Bc, Mrs/Ms).
# ==========================================
DUNLOP_CURVES = [
    (
        np.array([1.2587269, 1.4250513, 1.4435318, 1.4527721, 1.4620123, 1.4712526, 1.4804928, 1.5082136, 1.526694, 1.5359343, 1.5544148, 1.5728953, 1.600616, 1.637577, 1.6652977, 1.7022587, 1.7577002, 1.8039014, 1.8501027, 1.8870637, 1.9240246, 1.9517454, 1.9979466, 2.0441478, 2.0811088, 2.1273101, 2.164271, 2.2197125, 2.2843943, 2.3398357, 2.4045175, 2.4599589, 2.5431211, 2.6170431, 2.7094456, 2.801848, 2.8942505, 2.9958932, 3.1344969, 3.2823409, 3.4486653, 3.6149897, 3.8829569, 4.187885, 4.5020534, 4.7700205, 5.0102669], dtype=float),
        np.array([0.5, 0.37996219, 0.37334594, 0.36672968, 0.35916824, 0.35349716, 0.34499055, 0.33742911, 0.32986767, 0.32230624, 0.3147448, 0.30812854, 0.29678639, 0.28638941, 0.27599244, 0.2637051, 0.24669187, 0.2362949, 0.22400756, 0.21550095, 0.20699433, 0.20037807, 0.19187146, 0.18336484, 0.17674858, 0.16918715, 0.16257089, 0.15500945, 0.14650284, 0.1389414, 0.13232514, 0.12570888, 0.11814745, 0.11058601, 0.1020794, 0.095463138, 0.088846881, 0.082230624, 0.073724008, 0.06710775, 0.059546314, 0.052930057, 0.043478261, 0.034026465, 0.027410208, 0.02173913, 0.016068053], dtype=float),
    ),  # Curve 1
    (
        np.array([2.1919918, 2.1919918, 2.1919918, 2.2474333, 2.3028747, 2.349076, 2.4229979, 2.4969199, 2.5708419, 2.6355236, 2.7094456, 2.7833676, 2.8480493, 2.9312115, 3.0143737, 3.0790554, 3.2731006, 3.476386, 3.7073922, 3.9383984, 4.1601643, 4.2895277, 4.4281314, 4.5667351, 4.6960986, 4.825462, 6, 8, 10, 15, 20], dtype=float),
        np.array([0.41115312, 0.41115312, 0.41115312, 0.40642722, 0.4026465, 0.39886578, 0.39413989, 0.38846881, 0.38374291, 0.37901701, 0.37334594, 0.36956522, 0.36483932, 0.36011342, 0.35538752, 0.35160681, 0.34120983, 0.33081285, 0.3194707, 0.30907372, 0.29962193, 0.29395085, 0.28827977, 0.28355388, 0.27882798, 0.27410208, 0.235, 0.19, 0.16, 0.115, 0.091], dtype=float),
    ),  # Curve 2
    (
        np.array([1.2679671, 1.3234086, 1.3880903, 1.4804928, 1.5359343, 1.6190965, 1.6930185, 1.74846, 1.8039014, 1.8685832, 1.9332649, 2.0256674, 2.1273101, 2.2566735, 2.386037, 2.4876797, 2.5985626, 2.7094456, 2.8480493, 2.9681725, 3.0513347], dtype=float),
        np.array([0.5, 0.48960302, 0.47826087, 0.46691871, 0.4584121, 0.44990548, 0.44328922, 0.43950851, 0.43478261, 0.43005671, 0.42533081, 0.42060491, 0.41398866, 0.40831758, 0.4026465, 0.39886578, 0.39508507, 0.39224953, 0.38941399, 0.38657845, 0.38468809], dtype=float),
    ),  # Curve 3
]

# Global Plot Styling (Publication Quality)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.linewidth"] = 1.0
plt.rcParams["xtick.major.width"] = 1.0
plt.rcParams["ytick.major.width"] = 1.0


def smooth_dynamic(M, B, min_window=5, max_window=21, poly=3):
    """Adaptive Savitzky-Golay smoothing filter based on local gradient."""
    grad = np.abs(np.gradient(M, B))
    grad_norm = (grad - grad.min()) / (grad.max() - grad.min() + 1e-12)
    win_sizes = (max_window - min_window) * (1 - grad_norm) + min_window
    win_sizes = np.array([int(w) for w in win_sizes])
    M_smooth = np.zeros_like(M)
    N = len(M)
    for i, w in enumerate(win_sizes):
        w = max(w, poly + 2)
        half = w // 2
        lo = max(i - half, 0)
        hi = min(i + half + 1, N)
        wl = hi - lo
        if wl <= poly:
            M_smooth[i] = M[i]
            continue
        if wl % 2 == 0:
            wl -= 1
        wl = max(wl, poly + 1)
        poly_order = min(poly, wl - 1)
        M_smooth[i] = savgol_filter(M[lo:hi], wl, poly_order)[i - lo]
    return M_smooth


def parse_vsm_file(file_path):
    """Parses VSM data files, CSVs, or comma-delimited text files."""
    data_lines = []

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    for line in lines:
        line_s = line.strip()
        if not line_s or line_s.startswith("#"):
            continue
        parts = line_s.split(",")
        if len(parts) == 2:
            try:
                f_val = float(parts[0])
                m_val = float(parts[1])
                data_lines.append((f_val, m_val))
            except ValueError:
                continue

    if not data_lines:
        raise ValueError(f"No valid numerical data detected in: {file_path}")

    df = pd.DataFrame(data_lines, columns=["Field", "Moment"])
    return df


def process_hys(hys_path, sample_mass=None, cutoff_ratio=0.70):
    """Processes Hysteresis loop data."""
    df = parse_vsm_file(hys_path)
    fields = df["Field"].values
    moments = df["Moment"].values

    if sample_mass and sample_mass > 0:
        moments = moments / sample_mass

    max_field = np.max(fields)
    high_field_mask = fields >= (cutoff_ratio * max_field)

    if np.sum(high_field_mask) >= 3:
        chi, _ = np.polyfit(
            fields[high_field_mask], moments[high_field_mask], 1
        )
    else:
        chi = 0.0

    corrected_moments = moments - (chi * fields)

    return {"fields": fields, "moments": moments, "corrected": corrected_moments}


def process_back(back_path, sample_mass=None):
    """Processes backfield measurements and expresses field in mT."""
    df = parse_vsm_file(back_path)
    fields = df["Field"].values
    moments = df["Moment"].values

    if sample_mass and sample_mass > 0:
        moments = moments / sample_mass

    fields_abs = np.abs(fields)

    valid_mask = fields_abs > 1e-6
    bf_fields_raw = fields_abs[valid_mask]
    bf_moments = moments[valid_mask]

    sort_idx = np.argsort(bf_fields_raw)
    bf_fields_raw = bf_fields_raw[sort_idx]
    bf_moments = bf_moments[sort_idx]

    # Convert field to mT (assuming raw field is stored in T)
    bf_fields_T = bf_fields_raw * 1000.0

    # Dynamic smoothing over M
    bf_moments_smooth = smooth_dynamic(bf_moments, bf_fields_T)

    # Logarithmic derivative
    grad_raw = -np.log(10) * bf_fields_T * np.gradient(bf_moments, bf_fields_T)
    grad_smooth = (
        -np.log(10)
        * bf_fields_T
        * np.gradient(bf_moments_smooth, bf_fields_T)
    )

    return {
        "fields_T": bf_fields_T,
        "moments": bf_moments,
        "moments_smooth": bf_moments_smooth,
        "grad_raw": grad_raw,
        "grad_smooth": grad_smooth,
    }


def interpolate_zero_crossing(x, y):
    """Return linearly interpolated x positions where y crosses zero."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    crossings = []
    for i in range(len(y) - 1):
        y1, y2 = y[i], y[i + 1]
        x1, x2 = x[i], x[i + 1]
        if not np.isfinite([x1, x2, y1, y2]).all():
            continue
        if y1 == 0:
            crossings.append(x1)
        elif y1 * y2 < 0 and y2 != y1:
            crossings.append(x1 - y1 * (x2 - x1) / (y2 - y1))
    return np.asarray(crossings, dtype=float)


def hysteresis_day_parameters(data_hys, cutoff_ratio=0.70):
    """Estimate Ms, Mrs and Bc from the paramagnetically corrected hysteresis loop."""
    B = np.asarray(data_hys["fields"], dtype=float)
    M = np.asarray(data_hys["corrected"], dtype=float)
    finite = np.isfinite(B) & np.isfinite(M)
    B, M = B[finite], M[finite]
    if len(B) < 4:
        return {"Ms": np.nan, "Mrs": np.nan, "Bc": np.nan}

    max_abs_B = np.max(np.abs(B))
    sat_mask = np.abs(B) >= cutoff_ratio * max_abs_B
    Ms = np.nanmean(np.abs(M[sat_mask])) if np.any(sat_mask) else np.nan

    # Mrs: interpolate M where each loop branch crosses B = 0.
    m_at_zero = []
    for i in range(len(B) - 1):
        b1, b2 = B[i], B[i + 1]
        if b1 == 0:
            m_at_zero.append(M[i])
        elif b1 * b2 < 0 and b2 != b1:
            m0 = M[i] + (0.0 - b1) * (M[i + 1] - M[i]) / (b2 - b1)
            m_at_zero.append(m0)
    Mrs = np.nanmean(np.abs(m_at_zero)) if m_at_zero else np.nan

    # Bc: interpolate field positions where corrected M crosses zero.
    bc_crossings = interpolate_zero_crossing(B, M)
    if bc_crossings.size:
        # Reject possible high-field numerical crossings; coercive crossings are near the origin.
        bc_crossings = bc_crossings[np.abs(bc_crossings) <= 0.5 * max_abs_B]
    # Convert Bc from T to mT so it is in the same units as Bcr.
    Bc = np.nanmean(np.abs(bc_crossings)) * 1000.0 if bc_crossings.size else np.nan

    return {"Ms": Ms, "Mrs": Mrs, "Bc": Bc}


def backfield_bcr(data_back, use_smoothed=False):
    """
    Estimate Bcr (mT) by linear interpolation at the M_r = 0 crossing.

    The raw backfield curve is used by default. ``use_smoothed=True`` is kept
    only as a diagnostic to quantify whether smoothing moves the zero crossing.
    """
    B = np.asarray(data_back["fields_T"], dtype=float)
    m_key = "moments_smooth" if use_smoothed else "moments"
    M = np.asarray(data_back[m_key], dtype=float)
    finite = np.isfinite(B) & np.isfinite(M)
    B, M = B[finite], M[finite]
    crossings = interpolate_zero_crossing(B, M)
    crossings = crossings[crossings >= 0]
    return np.nanmin(crossings) if crossings.size else np.nan


def plot_day(ax, day_params, current_sample, dunlop_curves=None):
    """Plot Mrs/Ms versus Bcr/Bc with Dunlop mixing curves and sample data."""

    # ----------------------------------------------------
    # Dunlop mixing curves (background)
    # ----------------------------------------------------
    if dunlop_curves:
        for i, (x_curve, y_curve) in enumerate(dunlop_curves):
            ax.plot(
                x_curve,
                y_curve,
                color="0.60",
                lw=1.25,
                alpha=0.85,
                zorder=1,
                label="Dunlop mixing curves" if i == 0 else "_nolegend_",
            )

    # ----------------------------------------------------
    # All valid samples
    # ----------------------------------------------------
    xs, ys = [], []
    for sid, p in day_params.items():
        if (
            np.isfinite(p.get("Bc", np.nan))
            and np.isfinite(p.get("Bcr", np.nan))
            and np.isfinite(p.get("Ms", np.nan))
            and np.isfinite(p.get("Mrs", np.nan))
            and p["Bc"] > 0
            and p["Ms"] > 0
        ):
            x = p["Bcr"] / p["Bc"]
            y = p["Mrs"] / p["Ms"]
            if np.isfinite(x) and np.isfinite(y) and x > 0 and y > 0:
                xs.append(x)
                ys.append(y)

    if xs:
        ax.scatter(
            xs, ys, s=30, alpha=0.45, color="0.65", edgecolors="none",
            label="All samples", zorder=3
        )

    # ----------------------------------------------------
    # Current sample highlighted
    # ----------------------------------------------------
    if current_sample in day_params:
        p = day_params[current_sample]
        if p.get("Bc", np.nan) > 0 and p.get("Ms", np.nan) > 0:
            x = p["Bcr"] / p["Bc"]
            y = p["Mrs"] / p["Ms"]
            if np.isfinite(x) and np.isfinite(y) and x > 0 and y > 0:
                ax.scatter(
                    x, y, s=50, marker="o", color="#d62728", edgecolors="black",
                    linewidths=0.7, zorder=5, label=current_sample
                )

    # Day-plot axes: logarithmic Bcr/Bc and linear Mrs/Ms.
    ax.set_xscale("log")
    ax.set_xlim(1, 20)
    ax.set_ylim(0.0, 0.55)
    ax.set_yticks([0.0, 0.5])

    ax.set_title(f"Sample: {current_sample}\nDay Plot", fontsize=10.5, fontweight="bold", pad=10)
    ax.set_xlabel(r"$B_{cr}/B_c$", fontsize=9.0, fontweight="medium", labelpad=6)
    ax.set_ylabel(r"$M_{rs}/M_s$", fontsize=9.0, fontweight="medium", labelpad=6)

    # Grid styling similar to the reference Day plot.
    ax.grid(True, which="major", linestyle="--", color="#b0b0b0", alpha=0.9)
    ax.grid(True, which="minor", linestyle=":", color="#d0d0d0", alpha=0.8)
    ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=8.0)

    # Legend fixed at upper right.
    handles, legend_labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(
            handles, legend_labels, loc="upper right", frameon=True,
            facecolor="white", framealpha=0.9, fontsize=7.5
        )


def format_axis_style(ax, title, xlabel, ylabel, is_log_x=False, is_hys=False, y_data=None):
    """Applies a clean, elegant styling to plot axes."""
    ax.set_title(title, fontsize=10.5, fontweight="bold", pad=10)
    ax.set_xlabel(xlabel, fontsize=9.0, fontweight="medium", labelpad=6)
    ax.set_ylabel(ylabel, fontsize=9.0, fontweight="medium", labelpad=6)

    # Grid styling
    if is_log_x:
        ax.set_xscale("log")
        ax.grid(True, which="major", linestyle="-", color="#d3d3d3", alpha=0.7)
        ax.grid(
            True, which="minor", linestyle=":", color="#e0e0e0", alpha=0.5
        )
    else:
        ax.grid(True, linestyle="--", color="#d3d3d3", alpha=0.7)

    # Formateo de Notación Científica en el Eje Y
    ax.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
    ax.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))

    # Formateo específico para Histéresis
    if is_hys:
        # Eje X: fijar etiquetas exactas -1, -0.5, 0, 0.5, 1
        ax.set_xlim(-1, 1)
        ax.set_xticks([-1, -0.5, 0, 0.5, 1])
        ax.set_xticklabels(["-1", "-0.5", "0", "0.5", "1"])

        # Eje Y: simétrico centrado en 0 con sólo min, 0 y max
        if y_data is not None:
            y_max = max(abs(np.min(y_data)), abs(np.max(y_data)))
            ax.set_ylim(-y_max, y_max)
            ax.set_yticks([-y_max, 0, y_max])

    # Ticks formatting
    ax.tick_params(
        axis="both", which="both", direction="in", top=True, right=True, labelsize=8.0
    )



from pathlib import Path
def get_sample_mass(mass_dict, sample_id):
    return mass_dict.get(sample_id)


def execute_screening(samples_dict, mass_dict, run_dir, cutoff=.70, output_name=""):
    global PARAMAG_CUTOFF_RATIO, OUTPUT_FILENAME, BCR_OUTPUT_FILENAME
    PARAMAG_CUTOFF_RATIO = cutoff
    OUTPUT_FILENAME = str(run_dir/((output_name or 'VSM-summary')+'.pdf'))
    BCR_OUTPUT_FILENAME = str(run_dir/(output_name+'_Bcr.xlsx' if output_name else 'Bcr_from_backfield.xlsx'))
    sample_ids = sorted(samples_dict)
    num_samples = len(sample_ids)
    dunlop_curves = DUNLOP_CURVES
    # Pre-compute Day-plot parameters for every sample so each row can show the
    # complete sample population and highlight its own point.
    day_params = {}
    for sample_id in sample_ids:
        hys_path = samples_dict[sample_id]["hys"]
        back_path = samples_dict[sample_id]["back"]
        sample_mass = get_sample_mass(mass_dict, sample_id)
        params = {"Ms": np.nan, "Mrs": np.nan, "Bc": np.nan, "Bcr": np.nan}
        try:
            if hys_path:
                dh = process_hys(hys_path, sample_mass=sample_mass, cutoff_ratio=PARAMAG_CUTOFF_RATIO)
                params.update(hysteresis_day_parameters(dh, cutoff_ratio=PARAMAG_CUTOFF_RATIO))
            if back_path:
                db = process_back(back_path, sample_mass=sample_mass)
                params["Bcr"] = backfield_bcr(db)
        except Exception as e:
            print(f"⚠️ Could not calculate Day parameters for {sample_id}: {e}")
        day_params[sample_id] = params

    # ----------------------------------------------------
    # Bcr EXPORT FROM BACKFIELD CURVES
    # ----------------------------------------------------
    # Main Bcr is calculated from the RAW backfield remanence curve by linear
    # interpolation between the two measurements bracketing M_r = 0. A smoothed
    # value is also included as a diagnostic; the Day plot continues to use RAW Bcr.
    bcr_rows = []
    for sample_id in sample_ids:
        back_path = samples_dict[sample_id]["back"]
        if not back_path:
            continue

        sample_mass = get_sample_mass(mass_dict, sample_id)
        try:
            data_back = process_back(back_path, sample_mass=sample_mass)
            bcr_raw = backfield_bcr(data_back, use_smoothed=False)
            bcr_smooth = backfield_bcr(data_back, use_smoothed=True)

            treatment_match = re.search(r"_(ori|irr)$", sample_id, flags=re.IGNORECASE)
            treatment = treatment_match.group(1).lower() if treatment_match else ""
            pair_id = re.sub(r"_(ori|irr)$", "", sample_id, flags=re.IGNORECASE)

            bcr_rows.append({
                "Specimen": sample_id,
                "Pair_ID": pair_id,
                "Treatment": treatment,
                "Bcr_mT": float(bcr_raw) if np.isfinite(bcr_raw) else np.nan,
                "Bcr_smoothed_mT": float(bcr_smooth) if np.isfinite(bcr_smooth) else np.nan,
                "Delta_smoothed_minus_raw_mT": (
                    float(bcr_smooth - bcr_raw)
                    if np.isfinite(bcr_raw) and np.isfinite(bcr_smooth)
                    else np.nan
                ),
                "Backfield_file": os.path.basename(back_path),
            })
        except Exception as e:
            print(f"⚠️ Could not calculate/export Bcr for {sample_id}: {e}")

    if bcr_rows:
        bcr_df = pd.DataFrame(bcr_rows)
        bcr_df = bcr_df.sort_values(["Pair_ID", "Treatment", "Specimen"]).reset_index(drop=True)
        bcr_df.to_excel(BCR_OUTPUT_FILENAME, index=False)
        print(
            f"✅ Bcr values exported to '{BCR_OUTPUT_FILENAME}' "
            f"for {len(bcr_df)} backfield curve(s)."
        )
    else:
        print("⚠️ No Bcr values were exported.")

    if num_samples == 0:
        print(
            f"⚠️ No files found in '{RAW_DATA_DIR}'. Please check directory path."
        )
    else:
        # Set 5 columns layout: raw Hys, corrected Hys, backfield, gradient, Day plot
        fig, axes = plt.subplots(
            nrows=num_samples,
            ncols=5,
            figsize=(22, 3.8 * num_samples),
            squeeze=False,
        )

        for row_idx, sample_id in enumerate(sample_ids):
            hys_path = samples_dict[sample_id]["hys"]
            back_path = samples_dict[sample_id]["back"]
            sample_mass = get_sample_mass(mass_dict, sample_id)

            # Construcción del string de masa para el título
            if sample_mass is not None:
                mass_str = f" ({sample_mass * 1000:.3f} g)"
            else:
                mass_str = " (N/A mass)"

            # Labels depending on mass availability
            m_label = (
                "Magnetization, $M$ (Am²/kg)"
                if sample_mass
                else "Magnetic Moment, $M$ (Am²)"
            )
            mr_label = (
                "Remanence, $M_r$ (Am²/kg)"
                if sample_mass
                else "Remanence, $M_r$ (Am²)"
            )
            grad_label = (
                "$-dM_r / d(\\log_{10} B)$ (Am²/kg)"
                if sample_mass
                else "$-dM_r / d(\\log_{10} B)$ (Am²)"
            )

            # ----------------------------------------------------
            # 1. HYSTERESIS (Cols 1 & 2)
            # ----------------------------------------------------
            if hys_path:
                try:
                    data_hys = process_hys(
                        hys_path,
                        sample_mass=sample_mass,
                        cutoff_ratio=PARAMAG_CUTOFF_RATIO,
                    )

                    # Col 1: Raw Loop
                    axes[row_idx, 0].plot(
                        data_hys["fields"],
                        data_hys["moments"],
                        color="#1f77b4",
                        lw=1.4,
                    )
                    format_axis_style(
                        axes[row_idx, 0],
                        f"Sample: {sample_id}{mass_str}\nRaw Hysteresis Loop",
                        "Field, $B$ (T)",
                        m_label,
                        is_hys=True,
                        y_data=data_hys["moments"],
                    )

                    # Col 2: Corrected Loop
                    axes[row_idx, 1].plot(
                        data_hys["fields"],
                        data_hys["corrected"],
                        color="#2ca02c",
                        lw=1.4,
                    )
                    format_axis_style(
                        axes[row_idx, 1],
                        f"Sample: {sample_id}{mass_str}\nParamagnetic Corrected",
                        "Field, $B$ (T)",
                        m_label,
                        is_hys=True,
                        y_data=data_hys["corrected"],
                    )

                except Exception as e:
                    print(f"Error processing hysteresis for {sample_id}: {e}")
            else:
                for c in [0, 1]:
                    axes[row_idx, c].set_title(f"Sample: {sample_id}{mass_str}\n[Missing _hys file]")
                    axes[row_idx, c].axis("off")

            # ----------------------------------------------------
            # 2. BACKFIELD & GRADIENT (Cols 3 & 4)
            # ----------------------------------------------------
            if back_path:
                try:
                    data_back = process_back(back_path, sample_mass=sample_mass)

                    # Col 3: Backfield Curve
                    axes[row_idx, 2].plot(
                        data_back["fields_T"],
                        data_back["moments"],
                        color="#d62728",
                        lw=1.4,
                    )
                    format_axis_style(
                        axes[row_idx, 2],
                        f"Sample: {sample_id}{mass_str}\nBackfield Curve",
                        "Field, $B$ (mT)",
                        mr_label,
                    )

                    # Col 4: Full Gradient
                    axes[row_idx, 3].plot(
                        data_back["fields_T"],
                        data_back["grad_raw"],
                        color="#9467bd",
                        alpha=0.35,
                        lw=0.9,
                        label="Raw",
                    )
                    axes[row_idx, 3].plot(
                        data_back["fields_T"],
                        data_back["grad_smooth"],
                        color="#7b2cbf",
                        lw=1.8,
                        label="Smoothed",
                    )
                    format_axis_style(
                        axes[row_idx, 3],
                        f"Sample: {sample_id}{mass_str}\nGradient $-dM_r / d(\\log_{10} B)$",
                        "Field, $B$ (mT)",
                        grad_label,
                        is_log_x=True,
                    )
                    axes[row_idx, 3].legend(
                        loc="upper left", frameon=True, facecolor="white", framealpha=0.9, fontsize=8.0
                    )

                    # Col 5 is reserved for the Day plot (drawn below).

                except Exception as e:
                    print(f"Error processing backfield for {sample_id}: {e}")
            else:
                for c in [2, 3]:
                    axes[row_idx, c].set_title(f"Sample: {sample_id}{mass_str}\n[Missing _back file]")
                    axes[row_idx, c].axis("off")

            # ----------------------------------------------------
            # 3. DAY PLOT (Col 5)
            # ----------------------------------------------------
            plot_day(axes[row_idx, 4], day_params, sample_id, dunlop_curves)

        plt.tight_layout(pad=2.0)

        plt.savefig(OUTPUT_FILENAME, dpi=150, bbox_inches="tight")
        fig.savefig(str(Path(OUTPUT_FILENAME).with_suffix(".png")), dpi=150, bbox_inches="tight")
        print(f"💾 Figure successfully saved as '{OUTPUT_FILENAME}'.")

        plt.show()
        plt.close(fig)




## Interactive controls
Select paths, scan and review before processing. File processing text is shown separately from the mass ID.

In [ ]:
from pathlib import Path
import csv, io, json, hashlib, datetime
import ipywidgets as W
from IPython.display import display, clear_output

# ---- Appearance: edit these for minor style changes, then rerun the notebook. ----
plt.rcParams.update({'font.size': 10, 'lines.linewidth': 0.9})


def example_pattern(example):
    """Accept comma-separated examples, longest formats first to avoid truncation."""
    examples = list(dict.fromkeys(x.strip() for x in example.split(',') if x.strip()))
    if not examples:
        raise ValueError('Enter one or more comma-separated specimen IDs, or an explicit regex.')
    patterns=[]
    for value in sorted(examples, key=len, reverse=True):
        pattern=''.join('[A-Za-z]{%d}' % len(t) if t.isalpha() else
                        '[0-9]{%d}' % len(t) if t.isdigit() else re.escape(t)
                        for t in re.findall(r'[A-Za-z]+|[0-9]+|[^A-Za-z0-9]+',value))
        # Require an ID boundary for each alternative before accepting it.
        patterns.append('(?:'+pattern+r')(?=$|[_\-. ])')
    return '(?:'+'|'.join(dict.fromkeys(patterns))+')'


def split_filename(name, example, hys_suffixes, back_suffixes, regex=''):
    # An ending can be the extension itself or precede a conventional text extension.
    candidates = [(name, '')]
    p = Path(name)
    if p.suffix.lower() in ('.csv', '.txt', '.dat'):
        candidates.append((p.stem, p.suffix))
    hits = []
    for kind, endings in [('hys', hys_suffixes), ('back', back_suffixes)]:
        for suffix in (x.strip() for x in endings.split(',') if x.strip()):
            for stem, ext in candidates:
                if stem.casefold().endswith(suffix.casefold()):
                    hits.append((kind, stem[:-len(suffix)], suffix, ext))
    hits = list(dict.fromkeys(hits))
    if not hits:
        return None
    if len(hits) != 1:
        raise ValueError(f'Ambiguous measurement suffix: {name}')
    kind, stem, suffix, ext = hits[0]
    pattern = regex.strip() or example_pattern(example)
    match = re.match('(?:' + pattern + ')', stem)
    if not match or not match.group(0):
        raise ValueError(f'Cannot identify specimen in {name}; edit example/regex.')
    sample = match.groupdict().get('sample') or match.group(0)
    remainder = stem[match.end():]
    if remainder and remainder[0] not in '_-. ':
        raise ValueError(f'ID ends inside a token in {name}; check example/regex.')
    return dict(sample=sample, processing=remainder.lstrip('_-. '), series='',
                kind=kind, ending=suffix, extension=ext)


def read_masses(path, sheet, id_col, mass_col, unit):
    if not path.strip():
        return {}
    df = pd.read_excel(Path(path).expanduser(), sheet_name=sheet)
    if id_col not in df or mass_col not in df:
        raise ValueError('Choose specimen and mass columns from the Excel file.')
    result = {}
    factor = {'kg': 1., 'g': .001, 'mg': .000001}[unit]
    for _, row in df.iterrows():
        if pd.isna(row[id_col]) and pd.isna(row[mass_col]):
            continue
        if pd.isna(row[id_col]):
            raise ValueError('Mass row with missing specimen ID.')
        key = str(row[id_col]).strip()
        value = float(row[mass_col]) * factor
        if not key or not np.isfinite(value) or value <= 0:
            raise ValueError(f'Invalid mass for {key!r}.')
        if key in result:
            raise ValueError(f'Duplicate mass ID {key!r}; resolve in Excel first.')
        result[key] = value
    return result


def chooser(title, target, directories=True):
    # FileChooser is optional: manual path fields always remain usable.
    try:
        from ipyfilechooser import FileChooser
        fc = FileChooser(path=str(Path.cwd()), title=title, show_only_dirs=directories)
        def chosen(c):
            value = c.selected_path if directories else c.selected
            if value:
                target.value = value
        fc.register_callback(chosen)
        return fc
    except ImportError:
        return W.HTML('Optional browse button: install ipyfilechooser; otherwise paste a local path below.')

mode_w = W.Dropdown(options=[('Screening: figures and Bcr','screening'),
    ('Convert backfields only','convert')], value='screening', description='Mode:',
    layout=W.Layout(width='450px'))
conversion_units_w = W.Dropdown(options=[('Original magnetic moment (A m²)','moment'),
    ('Normalize by specimen mass (A m²/kg)','mass')], value='moment',
    description='Conversion:', layout=W.Layout(width='450px'))
input_path = W.Text(description='Input folder:', layout=W.Layout(width='95%'))
output_name_w = W.Text(description='Output name:', placeholder='Optional; blank keeps default names', layout=W.Layout(width='95%'))
output_path = W.Text(description='Save under:', layout=W.Layout(width='95%'))
examples = W.Text(value='TA03-02, CAS10-03', description='ID examples:')
regex_w = W.Text(description='ID regex:', placeholder='Optional; e.g. (?P<sample>[A-Z]{2}[0-9]{2}-[0-9]{2})', layout=W.Layout(width='95%'))
regex_help = W.Accordion(children=[W.HTML(value='<b>ID regex — copy one of these examples into the field above</b><table><tr><th>Format</th><th>Regex</th><th>Recognised IDs</th></tr><tr><td>Two letters + two digits</td><td><code>[A-Za-z]{2}[0-9]{2}</code></td><td>TA02, JO01</td></tr><tr><td>Two/three letters + two digits + hyphen + two digits</td><td><code>[A-Za-z]{2,3}[0-9]{2}-[0-9]{2}</code></td><td>TA03-02, CAS10-03</td></tr><tr><td>Optional hyphen and specimen number</td><td><code>[A-Za-z]{2,3}[0-9]{2}(?:-[0-9]{2})?</code></td><td>TA02, TA03-02, CAS10-03</td></tr><tr><td>Optional final specimen letter</td><td><code>[A-Za-z]{2,3}[0-9]{2}(?:-[0-9]{2}[A-Za-z]?)?</code></td><td>TA02, TA03-02A, CAS10-03</td></tr></table><p>Regex overrides ID examples. Leave it empty to use comma-separated examples instead. After changing it, click <b>Scan files</b> and check the <b>sample</b> column. IDs are read from the beginning of filenames; the measurement ending is configured separately.</p>')])
regex_help.set_title(0, "ID regex: examples and help")
regex_help.selected_index = None
hys_w = W.Text(value='_hys,.hys', description='Hys endings:')
back_w = W.Text(value='_back,.back,_backfield', description='DCD endings:')
recursive = W.Checkbox(value=False, description='Include subfolders')
mass_path = W.Text(description='Mass Excel:', layout=W.Layout(width='95%'))
sheet_w = W.Dropdown(options=[], description='Sheet:')
id_col_w = W.Dropdown(options=[], description='Specimen col:')
mass_col_w = W.Dropdown(options=[], description='Mass col:')
mass_unit_w = W.Dropdown(options=['g','mg','kg'], value='g', description='Mass unit:')
excel_btn = W.Button(description='Read Excel columns')
scan_btn = W.Button(description='1. Scan files')
run_btn = W.Button(description='2. Process selection', button_style='success')
files_w = W.SelectMultiple(options=[], description='Files:', rows=12, layout=W.Layout(width='95%'))
map_w = W.Textarea(description='Mapping CSV:', layout=W.Layout(width='95%', height='220px'))
cutoff_w = W.FloatSlider(value=.70, min=.3, max=.95, step=.05, description='High-field cut:')
out = W.Output()
STATE = {}


def refresh_columns(*args):
    if not mass_path.value.strip() or sheet_w.value is None:
        return
    df = pd.read_excel(Path(mass_path.value).expanduser(), sheet_name=sheet_w.value)
    cols = list(df.columns)
    id_col_w.options = cols
    preferred = next((c for c in cols if str(c).strip().casefold() == 'specimen_id'), None)
    if preferred is not None:
        id_col_w.value = preferred
    mass_col_w.options = cols
    if len(cols)>1:
        mass_col_w.value=cols[1]


def inspect_excel(_):
    with out:
        clear_output()
        try:
            book = pd.ExcelFile(Path(mass_path.value).expanduser())
            sheet_w.options = book.sheet_names
            sheet_w.value = book.sheet_names[0]
            refresh_columns()
            display(pd.read_excel(book, sheet_name=sheet_w.value).head())
            print('Choose the ID/mass columns and their unit explicitly. IDs are matched exactly (spaces trimmed).')
        except Exception as exc:
            print('Excel:', exc)

sheet_w.observe(refresh_columns, names='value')
excel_btn.on_click(inspect_excel)


def signature():
    return (input_path.value, examples.value, regex_w.value, hys_w.value, back_w.value, recursive.value, mode_w.value)


def scan(_=None):
    with out:
        clear_output()
        try:
            root = Path(input_path.value).expanduser()
            if not input_path.value.strip() or not root.is_dir():
                raise ValueError('Select an existing input folder.')
            rows, errors = [], []
            for path in sorted(root.rglob('*') if recursive.value else root.iterdir()):
                if not path.is_file():
                    continue
                try:
                    parts = split_filename(path.name, examples.value, hys_w.value, back_w.value, regex_w.value)
                    if parts and (mode_w.value != 'convert' or parts['kind'] == 'back'):
                        rows.append(dict(file=str(path.resolve()), **parts))
                except ValueError as exc:
                    errors.append(str(exc))
            if not rows:
                raise ValueError('No matching measurements. Check the endings and input folder. ' + '; '.join(errors))
            STATE.update(rows=rows, signature=signature())
            files_w.options = [(Path(r['file']).relative_to(root.resolve()).as_posix(), r['file']) for r in rows]
            files_w.value = tuple(r['file'] for r in rows)
            stream = io.StringIO(); writer=csv.DictWriter(stream, fieldnames=['file','sample','series','kind']);writer.writeheader()
            writer.writerows({k:r[k] for k in writer.fieldnames} for r in rows)
            map_w.value = stream.getvalue()
            display(pd.DataFrame(rows))
            print('Preview only: no files written. Select files and edit the mapping if needed.')
            print('Conversion mode keeps every selected backfield independently, including repeated specimens.' if mode_w.value == 'convert' else 'series separates treatments/repeats; pair Hys/DCD using the same sample and series.')
            if errors:
                print('Files needing attention (not selected):\n'+'\n'.join(errors))
        except Exception as exc:
            STATE.clear(); files_w.options=[]
            print('Scan:',exc)


def selected_mapping():
    if STATE.get('signature') != signature():
        raise ValueError('Input/naming options changed. Scan again.')
    selected=set(files_w.value); known={r['file'] for r in STATE['rows']}
    rows=list(csv.DictReader(io.StringIO(map_w.value)))
    result={}; masses_by_key={}; inventory=[]
    seen=set()
    for row in rows:
        path=row['file'].strip()
        if path not in selected:
            continue
        if path in seen or path not in known:
            raise ValueError('Duplicate or unknown file in mapping.')
        seen.add(path)
        sample=row['sample'].strip();series=row['series'].strip();kind=row['kind'].strip()
        if not sample or kind not in ('hys','back') or ' | ' in sample or ' | ' in series:
            raise ValueError('Invalid sample/series/kind in mapping.')
        if mode_w.value == 'convert' and kind != 'back':
            raise ValueError('Conversion mode accepts backfield files only. Scan again.')
        key=path if mode_w.value == 'convert' else sample + (' | '+series if series else '')
        entry=result.setdefault(key,{'hys':None,'back':None})
        if entry[kind]:
            raise ValueError(f'Multiple {kind} files for {key}. Deselect repeats or assign distinct series (with matching Hys/DCD).')
        entry[kind]=path;masses_by_key[key]=sample
        inventory.append(dict(row, file=path, sample=sample, series=series, kind=kind))
    if seen!=selected or not result:
        raise ValueError('Mapping must contain every selected file exactly once.')
    return result,masses_by_key,inventory


def output_basename():
    value=output_name_w.value.strip()
    if value in ('.','..') or any(c in value for c in '/\\:*?"<>|'):
        raise ValueError('Output name must be a simple filename, without folders or reserved characters.')
    return value


def run(_=None):
    with out:
        clear_output()
        try:
            samples,mass_ids,inventory=selected_mapping()
            if mode_w.value == 'convert':
                return convert_backfields_only(samples, mass_ids, inventory)
            masses=read_masses(mass_path.value,sheet_w.value,id_col_w.value,mass_col_w.value,mass_unit_w.value)
            if not output_path.value.strip() or not Path(output_path.value).expanduser().is_dir():
                raise ValueError('Choose an existing output parent folder.')
            missing=[k for k in samples if mass_ids[k] not in masses]
            # Parse inputs before creating a destination or exporting results.
            for entry in samples.values():
                for path in entry.values():
                    if path:
                        df=parse_vsm_file(path)
                        if len(df)<5 or not np.isfinite(df.to_numpy()).all():
                            raise ValueError(f'Invalid or too few data points: {path}')
            run_dir=Path(output_path.value).expanduser().resolve()
            mapped_masses={k:masses[mass_ids[k]] for k in samples if mass_ids[k] in masses}
            if missing:
                print('No matching mass (raw A m² retained):', ', '.join(missing))
            execute_screening(samples,mapped_masses,run_dir,cutoff_w.value,output_basename())
            print('Results:',run_dir)
        except Exception as exc:
            print('Processing stopped:',exc)

def convert_backfields_only(samples, mass_ids, inventory):
    """Independent measurements: grouping by specimen must never discard acquisitions."""
    if not output_path.value.strip() or not Path(output_path.value).expanduser().is_dir():
        raise ValueError('Choose an existing output parent folder.')
    normalize = conversion_units_w.value == 'mass'
    masses = read_masses(mass_path.value,sheet_w.value,id_col_w.value,mass_col_w.value,mass_unit_w.value) if normalize else {}
    prefix=output_basename()
    planned=[]
    names=set()
    for key,entry in samples.items():
        path=Path(entry['back'])
        # Retain the entire measurement basename, including acquisition settings
        # and the backfield marker; remove only a conventional storage extension.
        stem=path.stem if path.suffix.lower() in ('.csv','.txt','.dat','.back') else path.name
        name=(prefix+'_' if prefix else '')+stem+'_unmix.back'
        if name.casefold() in names:
            raise ValueError(f'Output filename collision: {name}. Rename identical basenames or process separately.')
        names.add(name.casefold())
        df=parse_vsm_file(path)
        if len(df)<2 or not np.isfinite(df.to_numpy()).all():
            raise ValueError(f'Invalid numeric data: {path}')
        field=np.abs(df.Field.to_numpy(float))*1000.
        if np.any(np.diff(field)<-1e-9):
            raise ValueError(f'Non-monotonic reverse-field magnitude: {path}. Inspect before conversion.')
        mass=masses.get(mass_ids[key])
        if normalize and (mass is None or mass<=0):
            raise ValueError(f'Missing mass for {mass_ids[key]}; select the mass Excel or use original moment.')
        signal=df.Moment.to_numpy(float)/(mass if normalize else 1.)
        table=pd.DataFrame({'Field_mT':field,
                            'Magnetization_Am2_kg' if normalize else 'Moment_Am2':signal})
        planned.append((path,name,table,mass,mass_ids[key]))
    # Validate all measurements before creating a run folder.
    dest=Path(output_path.value).expanduser().resolve()
    (dest/'converted').mkdir(parents=True, exist_ok=True);(dest/'tables').mkdir(exist_ok=True)
    rows=[]
    for path,name,table,mass,sample in planned:
        table.to_csv(dest/'converted'/name,index=False,float_format='%.16e')
        rows.append(dict(input_file=str(path),output_file=name,specimen_id=sample,
                         mass_kg=mass if normalize else None,signal_unit='A m²/kg' if normalize else 'A m²',
                         n_points=len(table),sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    pd.DataFrame(rows).to_csv(dest/'tables'/'conversion_inventory.csv',index=False)
    pd.DataFrame(inventory).to_csv(dest/'tables'/'input_mapping.csv',index=False)
    settings=dict(mode='convert_backfields_only',input_field_unit='T',output_field_unit='mT',
                  output_prefix=prefix,signal_unit='A m²/kg' if normalize else 'A m²',mass_file=mass_path.value if normalize else None,
                  mass_sheet=sheet_w.value if normalize else None,
                  mass_id_column=str(id_col_w.value) if normalize else None,
                  mass_column=str(mass_col_w.value) if normalize else None,
                  mass_unit=mass_unit_w.value if normalize else None,
                  preprocessing='Absolute reverse field T→mT only; no interpolation, smoothing, offsets or averaging. All input rows retained.')
    (dest/'settings.json').write_text(json.dumps(settings,indent=2))
    display(pd.DataFrame(rows))
    print(f'Converted {len(rows)} independent backfield measurements. No screening plots or Bcr calculation.')
    print('Results:',dest)


scan_btn.on_click(scan);run_btn.on_click(run)
display(W.VBox([
    mode_w, conversion_units_w,
    W.HTML('Convert backfields only: one output per selected measurement; acquisition parameters remain in filenames. Original moment is preserved by default; mass normalization is optional.'),
    W.HTML('<h3>1 · Input and specimen naming</h3>'),chooser('Select input folder',input_path),input_path,
    W.HBox([examples,hys_w,back_w]),regex_w,regex_help,recursive,
    W.HTML('Examples = specimen IDs separated by commas, e.g. TA03-02, CAS10-03 or JO01, TA03-02. Letter/digit counts and separators are inferred. '
           'Processing/acquisition text is shown separately. For variable formats use an explicit regex, optionally with a named sample group.'),
    scan_btn,files_w,map_w,
    W.HTML('<h3>2 · Optional masses</h3>'),chooser('Select mass Excel',mass_path,False),mass_path,
    excel_btn,W.HBox([sheet_w,id_col_w,mass_col_w,mass_unit_w]),
    W.HTML('<h3>3 · Output and processing</h3>'),chooser('Select output parent',output_path),output_path,output_name_w,
    W.HTML('Screening: custom name.pdf/.png and custom name_Bcr.xlsx. Conversion: custom name is a prefix; measurement names remain distinct. Blank keeps all default names.'),
    cutoff_w,run_btn,out]))
